# 04 · On-policy distillation: OPD → OPSD

Keep RL's on-policy sampling, but replace the 0/1 reward with a **per-token grade
from a teacher**:

    advantage[t] = log p_teacher(token_t) − log p_student(token_t)      (= −reverse KL)

The student samples, the teacher says, at every token, "how likely would *I* have
been to write that?" Same `importance_sampling` loss as RL, but the signal is dense.

Two variants. **The loop is identical; only the teacher changes:**

| | teacher | teacher sees |
|---|---|---|
| **OPD**  | bigger model | the same prompt + "be concise" |
| **OPSD** | *the student itself* (frozen copy) | the prompt **+ the reference solution** |

In [1]:
from common import *

GROUP_SIZE, BATCH_SIZE, STEPS, LR = 4, 16, 15, 1e-4
sampling_params = types.SamplingParams(max_tokens=MAX_TOKENS, temperature=1.0, stop=renderer.get_stop_sequences())
START_FROM = load_checkpoint("sft")
gsm8k_train = load_gsm8k("train").shuffle(seed=2)

/home/ray/default/post-training-webinar/.venv/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [2]:
# ── LIVE: the generic on-policy distillation loop ─────────────────────────
def on_policy_distill(get_batch, label, steps=STEPS):
    """get_batch(step) -> list of (student_prompt, teacher_client, teacher_prompt)."""
    training_client = new_training_client(START_FROM)
    for step in range(steps):
        student = training_client.save_weights_and_get_sampling_client().result()
        items = get_batch(step)
        sample_futs = [student.sample(sp, GROUP_SIZE, sampling_params) for sp, _, _ in items]

        # Teacher scores the student's tokens. It doesn't generate anything.
        scored = []
        for (student_prompt, teacher, teacher_prompt), fut in zip(items, sample_futs):
            for s in fut.result().sequences:
                full = teacher_prompt.append(types.EncodedTextChunk(tokens=s.tokens))
                tfut = teacher.compute_logprobs(full, prompt_logprobs_last_n=len(s.tokens))
                scored.append((student_prompt, s, tfut))

        batch, kls = [], []
        for student_prompt, s, tfut in scored:
            teacher_lp = tfut.result()[-len(s.tokens):]
            advantages = [t - st for t, st in zip(teacher_lp, s.logprobs)]  # per-token!
            kls.append(-sum(advantages) / len(advantages))
            batch.append(make_rl_datum(student_prompt, s.tokens, s.logprobs, advantages))

        training_client.forward_backward(batch, loss_fn="importance_sampling")
        training_client.optim_step(adam(LR)).result()
        print(f"[{label}] step {step:2d}  reverse KL to teacher = {sum(kls) / len(kls):.3f}")
    return training_client


def gsm8k_rows(step):
    return gsm8k_train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))


def student_prompt(question):
    return renderer.build_generation_prompt(gsm8k_convo(question))


def concise_teacher_prompt(question):
    # The teacher is *told* to be brief, so it gives high logprobs to short, direct solutions.
    # Without this, a chatty teacher would pull the student past our 256-token budget.
    convo = gsm8k_convo(question)
    convo[-1] = {**convo[-1], "content": convo[-1]["content"] + " Be concise: solve it in a few short sentences."}
    return renderer.build_generation_prompt(convo)

## OPD: on-policy distillation from a bigger teacher

The student sees the normal prompt; the teacher sees it plus "be concise".

In [3]:
# ── LIVE ──────────────────────────────────────────────────────────────────
math_teacher = service_client.create_sampling_client(base_model=MATH_TEACHER).result()

def opd_batch(step):
    return [(student_prompt(q), math_teacher, concise_teacher_prompt(q)) for q in gsm8k_rows(step)["question"]]

tc = on_policy_distill(opd_batch, "OPD")
eval_gsm8k(tc.save_weights_and_get_sampling_client().result(), label="4a. OPD (35B-A3B teacher)")

[OPD] step  0  reverse KL to teacher = 0.437
[OPD] step  1  reverse KL to teacher = 0.354
[OPD] step  2  reverse KL to teacher = 0.291
[OPD] step  3  reverse KL to teacher = 0.295
[OPD] step  4  reverse KL to teacher = 0.254
[OPD] step  5  reverse KL to teacher = 0.241
[OPD] step  6  reverse KL to teacher = 0.258
[OPD] step  7  reverse KL to teacher = 0.247
[OPD] step  8  reverse KL to teacher = 0.192
[OPD] step  9  reverse KL to teacher = 0.148
[OPD] step 10  reverse KL to teacher = 0.144
[OPD] step 11  reverse KL to teacher = 0.198
[OPD] step 12  reverse KL to teacher = 0.185
[OPD] step 13  reverse KL to teacher = 0.169
[OPD] step 14  reverse KL to teacher = 0.169
GSM8K accuracy within 256 tokens [4a. OPD (35B-A3B teacher)]: 89.0% (89/100)   [0 ran out of tokens]


0.89

## OPSD: on-policy *self*-distillation

No bigger model. The teacher is a frozen copy of the student, but it gets to
**peek at the reference solution** (privileged information). The student learns
to produce, without the answer key, what it would have written *with* it.

In [4]:
# ── LIVE: only the teacher + teacher prompt change ────────────────────────
self_teacher = new_training_client(START_FROM).save_weights_and_get_sampling_client().result()

def teacher_prompt_with_solution(question, answer):
    hint = f"{question}\n\n(Reference solution, for you only: {gsm8k_solution(answer)})"
    return renderer.build_generation_prompt([*FEWSHOT, {"role": "user", "content": hint + QUESTION_SUFFIX}])

def opsd_batch(step):
    rows = gsm8k_rows(step)
    return [
        (student_prompt(q), self_teacher, teacher_prompt_with_solution(q, a))
        for q, a in zip(rows["question"], rows["answer"])
    ]

tc = on_policy_distill(opsd_batch, "OPSD")
eval_gsm8k(tc.save_weights_and_get_sampling_client().result(), label="4b. OPSD (self + answer key)")

[OPSD] step  0  reverse KL to teacher = 0.049
[OPSD] step  1  reverse KL to teacher = 0.057
[OPSD] step  2  reverse KL to teacher = 0.045
[OPSD] step  3  reverse KL to teacher = 0.053
[OPSD] step  4  reverse KL to teacher = 0.049
[OPSD] step  5  reverse KL to teacher = 0.052
[OPSD] step  6  reverse KL to teacher = 0.065
[OPSD] step  7  reverse KL to teacher = 0.055
[OPSD] step  8  reverse KL to teacher = 0.054
[OPSD] step  9  reverse KL to teacher = 0.050
[OPSD] step 10  reverse KL to teacher = 0.065
[OPSD] step 11  reverse KL to teacher = 0.074
[OPSD] step 12  reverse KL to teacher = 0.054
[OPSD] step 13  reverse KL to teacher = 0.053
[OPSD] step 14  reverse KL to teacher = 0.060
GSM8K accuracy within 256 tokens [4b. OPSD (self + answer key)]: 76.0% (76/100)   [2 ran out of tokens]


0.76

## Scoreboard

In [5]:
scoreboard()

  0. base model                         55.0%  ███████████████████████████
  2. SFT                                78.0%  ███████████████████████████████████████
  3. RL (from SFT)                      87.0%  ███████████████████████████████████████████
  4a. OPD (35B-A3B teacher)             89.0%  ████████████████████████████████████████████
  4b. OPSD (self + answer key)          76.0%  ██████████████████████████████████████
  SkyRL · 3. RL                         89.0%  ████████████████████████████████████████████


**Takeaways**
- Distillation = RL where the reward is *dense* (every token) and comes from a model, not a verifier.
- OPD needs a stronger teacher. OPSD needs only privileged info (answer key, docs, tools).
- In Tinker, both were the *same 25-line loop*, swapping only `(teacher, teacher_prompt)`.